# 07 — RNN, LSTM, GRU

Recurrent layers for sequential data (text, time series, audio). LSTM/GRU address the vanishing-gradient problem of vanilla RNNs.

## Basic RNN

In [ ]:
import torch
import torch.nn as nn

rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=2, batch_first=True)
x = torch.randn(8, 5, 10)   # (batch, seq_len, input_size)
output, hidden = rnn(x)
output.shape   # (8, 5, 20) — output at every timestep
hidden.shape   # (2, 8, 20) — final hidden state per layer


## LSTM

In [ ]:
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True, bidirectional=False)
x = torch.randn(8, 5, 10)
output, (h_n, c_n) = lstm(x)
output.shape   # (8, 5, 20)
h_n.shape      # (2, 8, 20) — final hidden state
c_n.shape      # (2, 8, 20) — final cell state


## GRU (simpler gating, often comparable performance to LSTM)

In [ ]:
gru = nn.GRU(input_size=10, hidden_size=20, num_layers=1, batch_first=True)
output, h_n = gru(x)
output.shape


## Bidirectional RNN

In [ ]:
bilstm = nn.LSTM(input_size=10, hidden_size=20, batch_first=True, bidirectional=True)
output, _ = bilstm(x)
output.shape   # (8, 5, 40) — hidden_size * 2 for concatenated forward/backward


## Sequence Classification (many-to-one)

In [ ]:
class SeqClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)              # (batch, seq_len, embed_dim)
        _, (h_n, _) = self.lstm(x)         # h_n: (1, batch, hidden_dim)
        return self.fc(h_n.squeeze(0))     # use final hidden state for classification

model = SeqClassifier(vocab_size=5000, embed_dim=128, hidden_dim=256, num_classes=2)
tokens = torch.randint(1, 5000, (8, 20))
model(tokens).shape   # (8, 2)


## Variable-Length Sequences: Padding & Packing

Use `pack_padded_sequence` so the RNN skips padded timesteps instead of wasting compute on them.

In [ ]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

seqs = [torch.randn(3, 10), torch.randn(5, 10), torch.randn(2, 10)]
lengths = torch.tensor([len(s) for s in seqs])

padded = pad_sequence(seqs, batch_first=True)   # (3, 5, 10) — padded to max length
padded.shape

# sort by length descending (required by pack_padded_sequence unless enforce_sorted=False)
lengths_sorted, sort_idx = lengths.sort(descending=True)
padded_sorted = padded[sort_idx]

packed = pack_padded_sequence(padded_sorted, lengths_sorted, batch_first=True)
lstm_out, (h_n, c_n) = nn.LSTM(10, 16, batch_first=True)(packed)
unpacked, out_lengths = pad_packed_sequence(lstm_out, batch_first=True)
unpacked.shape


## Many-to-Many (sequence labeling, e.g. POS tagging)

In [ ]:
class SeqTagger(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)          # (batch, seq_len, hidden*2)
        return self.fc(out)            # (batch, seq_len, num_tags) — a prediction per timestep

model = SeqTagger(vocab_size=5000, embed_dim=64, hidden_dim=128, num_tags=10)
model(torch.randint(0, 5000, (4, 15))).shape   # (4, 15, 10)


## Time Series Forecasting

In [ ]:
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])   # predict from last timestep's hidden state

model = TimeSeriesLSTM()
window = torch.randn(16, 30, 1)   # 16 samples, 30 timesteps, 1 feature
model(window).shape   # (16, 1) — next-step prediction


## Common Gotchas
- `batch_first=True` is essential for consistency with DataLoader output `(batch, seq, feature)` — default is `(seq, batch, feature)`.
- Padding index in `nn.Embedding` (`padding_idx=0`) prevents the padding token from being updated during training.
- For classification from an RNN, use the final hidden state (`h_n`), not `output[-1]`, when sequences are padded (they may differ).
- LSTMs/RNNs are inherently sequential — much slower than CNNs/Transformers on GPU; consider Transformer-based approaches for long sequences.